# Eksperimen 4: Model Leksikal (Fitur Morfologis Sastrawi dan SVR)

**GEMASTIK KTI 2026** | Tim Peneliti

Eksperimen ini mengevaluasi 11 fitur linguistik sederhana yang digabungkan dengan algoritma SVR. Jika model regresi konvensional ini mampu mengimbangi arsitektur IndoBERT yang memiliki 110 juta parameter, hal tersebut akan menjadi bukti empiris yang valid bahwa morfologi Bahasa Indonesia memiliki sinyal penilaian otomatis yang sangat kuat.

## 0. Persiapan Lingkungan dan Konfigurasi

In [ ]:
import sys
import os

try:
    import google.colab
    IN_COLAB = True
    print("Lingkungan Eksekusi: Google Colab")
    if not os.path.exists("/content/indo-asag"):
        os.system("git clone https://github.com/Rnov24/indo-asag.git /content/indo-asag")
    os.system("pip install -q -e /content/indo-asag[all]")
    REPO_ROOT = "/content/indo-asag"
except ImportError:
    IN_COLAB = False
    try:
        REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
    except NameError:
        REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    print(f"Lingkungan Eksekusi: Lokal ({REPO_ROOT})")

sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from indo_asag.data import load_dataset
from indo_asag.features import extract_features_df, get_feature_names
from indo_asag.evaluation import run_multi_seed
from indo_asag.utils import set_seed, load_config
from indo_asag.utils.github import auto_push_to_github

config = load_config(os.path.join(REPO_ROOT, "configs", "base.yaml"))
SEEDS = config["seeds"]
RESULTS_DIR = os.path.join(REPO_ROOT, "results", "prelim")
PREDS_DIR = os.path.join(RESULTS_DIR, "predictions")

## 1. Pemuatan Dataset dan Ekstraksi Fitur Sastrawi

In [ ]:
DATA_PATH = os.path.join(REPO_ROOT, config["data"]["path"])
df = load_dataset(DATA_PATH)

print("\nMengekstrak 11 fitur Sastrawi (Handcrafted)...")
feat_df = extract_features_df(df)
hc_cols = get_feature_names()

for col in hc_cols:
    df[col] = feat_df[col]

# Save the extracted features so we don't have to extract them again in Exp 5
df[hc_cols].to_csv(os.path.join(PREDS_DIR, "features_hc.csv"), index=False)
print("[OK] Fitur berhasil diekstrak dan disimpan.")

## 2. Eksekusi Eksperimen 4

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

print("\n" + "=" * 60)
print("EXP 4: Model Leksikal (Fitur Morfologis Sastrawi dan SVR)")
print("=" * 60)

hc_oof_preds = {s: np.zeros(len(df)) for s in SEEDS}

def exp4_predict(train_df, val_df, fold, seed=42):
    X_tr = train_df[hc_cols].values
    X_va = val_df[hc_cols].values
    
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_va_s = sc.transform(X_va)
    
    svr = SVR(
        kernel=config["model"]["svr"]["kernel"],
        C=config["model"]["svr"]["C"],
        gamma=config["model"]["svr"]["gamma"],
        epsilon=config["model"]["svr"]["epsilon"]
    )
    svr.fit(X_tr_s, train_df["score_norm"].values)
    
    preds = svr.predict(X_va_s)
    hc_oof_preds[seed][val_df.index] = preds
    return preds

exp4_summary = run_multi_seed(df, exp4_predict, seeds=SEEDS)
print(f"  QWK: {exp4_summary['QWK']}")

## 3. Penyimpanan Prediksi (Out-of-Fold)

In [ ]:
print("\nMenyimpan array prediksi OOF ke disk...")
for s in SEEDS:
    np.save(os.path.join(PREDS_DIR, f"exp4_hc_oof_seed{s}.npy"), hc_oof_preds[s])
print("[OK] Prediksi berhasil disimpan.")

## 4. Publikasi Otomatis ke GitHub

In [ ]:
auto_push_to_github("chore(auto): menyimpan prediksi Eksperimen 4 Handcrafted SVR", IN_COLAB, REPO_ROOT)